# 052 — Optimizadores, regularización y schedulers

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** m = 0.1·(−0.4) = −0.04; v = 0.001·0.16 = 0.00016.
m̂ = −0.04/0.1 = −0.4; v̂ = 0.00016/0.001 = 0.16; √v̂ = 0.4.
Δθ = −0.01·(−0.4)/0.4 = **+0.01** → θ = **2.01**. Primer paso de magnitud
exactamente α, en dirección opuesta al gradiente: la corrección de sesgo "des-inicializa"
las medias móviles que arrancan en 0.

**Ejercicio 2.** Momento: v₁ = 1 → θ = −0.1; v₂ = 1.9 → θ = −0.29; v₃ = 2.71 →
θ = **−0.561**. SGD: 3 pasos de −0.1 → θ = **−0.3**. Con gradiente constante, el
momento acumula velocidad (límite geométrico: v → g/(1−μ) = 10, ¡pasos 10× mayores!).

**Ejercicio 3.** Entrenamiento: h/(1−p) = h/0.75 con máscara → (13.33, 26.67, 0, 53.33).
Inferencia: (10, 20, 30, 40). Esperanza en entrenamiento: cada unidad sobrevive con
probabilidad 0.75 y se escala por 1/0.75, luego E[salida] = 0.75·h/0.75 = h. El escalado
hace innecesario tocar nada en inferencia.

**Ejercicio 4.** t = 50: mitad del warmup → η = **0.0005**. t = 100: fin del warmup →
η = **0.001**. t = 550: η = 0.5·0.001·(1+cos(π·450/900)) = 0.5·0.001·(1+cos(π/2)) =
0.5·0.001·1 = **0.0005**.


In [ ]:
result = run_lab("optimization", seed=52)
assert result["kind"] == "optimization"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica
import math

# Ejercicio 1
theta, g, alpha, b1, b2, eps = 2.0, -0.4, 0.01, 0.9, 0.999, 1e-8
m = (1 - b1) * g
v = (1 - b2) * g * g
m_hat = m / (1 - b1)
v_hat = v / (1 - b2)
theta_new = theta - alpha * m_hat / (math.sqrt(v_hat) + eps)
print(f"m={m:.4f} v={v:.6f} m̂={m_hat:.4f} v̂={v_hat:.4f} θ={theta_new:.6f}")
assert abs(theta_new - 2.01) < 1e-6

# Ejercicio 2
v, theta = 0.0, 0.0
for _ in range(3):
    v = 0.9 * v + 1.0
    theta -= 0.1 * v
print("momento θ =", round(theta, 4), "| SGD θ =", -0.3)
assert abs(theta - (-0.561)) < 1e-9

# Ejercicio 4
def eta(t):
    if t <= 100:
        return 0.001 * t / 100
    return 0.5 * 0.001 * (1 + math.cos(math.pi * (t - 100) / 900))
print("η(50) =", eta(50), "| η(103) =", eta(103), "| η(550) =", round(eta(550), 6))


## Reflexión

1. ¿Por qué el primer paso de Adam tiene magnitud ≈ α sin importar la escala del gradiente, y qué papel exacto juega la corrección de sesgo?
2. Si duplicas el tamaño de batch manteniendo todo lo demás, ¿qué le pasa al ruido del gradiente y por qué suele acompañarse de un ajuste de η?
3. ¿Qué evidencia (curvas de train/validación) te haría elegir añadir dropout frente a añadir weight decay, o simplemente parar antes?
